In [1]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    temperature = 0,
    groq_api_key = '<your_api_here>',
    model_name = "llama-3.3-70b-versatile"
)

#response = llm.invoke("the first person to find india was ...")
#print(response.content)

In [3]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.naukri.com/software-developer-jobs")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
page_data = loader.load().pop().page_content
print(page_data)

In [5]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in json format containing the following keys:
    `role`, `experience` , `skills` and `description`.
    Only return the valid JSON.
    return at least 50 job postings.
    ### VALID JSON (NO PREAMBLE):
    """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

In [ ]:
json_res

In [8]:
import PyPDF2

pdf_path = "/Users/rutvij/Downloads/cold_email_generation_tool/de_samplecv.pdf"
text = ""

with open(pdf_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        text += page.extract_text()


In [27]:
from sentence_transformers import SentenceTransformer

/Users/rutvij/ai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
import numpy as np
model = SentenceTransformer('all-MiniLM-L6-v2')
resume_embedding = model.encode([text])
np.save("resume_embedding.npy", resume_embedding)

In [30]:
import chromadb

In [32]:
# Initialize Chroma (persistent recommended)
chroma_client = chromadb.PersistentClient(path="./chroma_store")
collection = chroma_client.get_or_create_collection(
    name="job_postings",
    metadata={"hnsw:space": "cosine"}
)


# ---  Load or compute resume embedding ---
RESUME_EMB_PATH = "resume_embedding.npy"

In [35]:
# Prepare text input for embeddings
job_texts = [
    f"{j['role']} {j['experience']} {j['skills']} {j['description']}"
    for j in json_res
]

# Encode all job postings
job_embeddings = model.encode(job_texts, show_progress_bar=True)

# Create unique IDs (if not present)
ids = [f"job_{i}" for i in range(len(json_res))]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.37it/s]


In [37]:
import json

def sanitize_metadata(metadata_list):
    sanitized = []
    for md in metadata_list:
        new_md = {}
        for k, v in md.items():
            if isinstance(v, list):
                new_md[k] = json.dumps(v)  # Convert lists to JSON strings
            else:
                new_md[k] = v
        sanitized.append(new_md)
    return sanitized

# Usage
metadatas = sanitize_metadata(json_res)

# Now you can safely add to Chroma
collection.add(
    ids=ids,
    documents=job_texts,
    metadatas=metadatas,
    embeddings=job_embeddings
)

In [38]:

print(f"Stored {len(json_res)} job postings in ChromaDB.")

Stored 55 job postings in ChromaDB.


In [69]:
results = collection.query(
    query_embeddings=resume_embedding,
    n_results=5
)
best_match = [md[0] for md in results['metadatas']]
job_role_name = best_match[0]['role']
job_role_desc = best_match[0]['description']

In [70]:
print(job_role_name)

Data Engineer


In [71]:
print(job_role_desc)

Design, build, and maintain large-scale data systems


In [73]:
type(job_role_desc)

str

In [80]:
prompt_email = PromptTemplate.from_template(
    """
    ### job role
    {job_role_name}
    #### job description
    {job_role_desc}
    
    
    ### INSTRUCTION:
    Your job is to write a cold email to the client regarding the job mentioned above how much it matches with the skills sets from the 
    candidate from resume
            Do not provide a preamble.
            ### EMAIL (NO PREAMBLE):
    ### VALID JSON (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm
res2 = chain_email.invoke({"job_role_name": str(job_role_name),"job_role_desc": str(job_role_desc)})
print(res2.content)

Subject: Expert Data Engineer for Large-Scale Data Systems

Dear Hiring Manager,

I came across the Data Engineer position at your esteemed organization and was impressed by the job description. As a seasoned data professional with a strong background in designing, building, and maintaining large-scale data systems, I believe I would be an excellent fit for this role.

Upon reviewing the job requirements, I noticed that my skills and experience align closely with the position. My expertise in data engineering, data architecture, and data warehousing would enable me to make significant contributions to your team. I have a proven track record of successfully building and maintaining large-scale data systems, ensuring data quality, and optimizing data processing workflows.

Some of the key skills that I bring to the table include:

* Data engineering: I have extensive experience in designing, building, and maintaining large-scale data systems, including data pipelines, data warehouses, an